# Concept
1. The program will execute a large number of SQL files in sequence.
2. Each SQL statement is CREATE ONLY.
3. Parameters such as `catalog_env`, `env`, and `schema_name` are provided as input.
4. Each SQL CREATE statement is saved as `<table_name>.sql` in the `datascripts` folder.
5. The program executes these scripts one after another.

In [0]:
from typing import Dict, List, Optional, Any
import os
import re
from pathlib import Path

In [0]:
%run "../helpers/000-log-helper"

In [0]:
# Get the name of the notebook
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
nb_name = nb_path.split("/")[-1]

# Instantiate the helper class to get the Spark session and logger
helper = HELPER()
spark = helper.get_spark_session(nb_name)
logger = helper.get_logger(nb_name)

In [0]:
notebook_path = "/Workspace/Repos/development/datavault-datamodelling/utils/generators_ddl_sql_statements/250-create-sql-statements-metadataregistry"

sql_files_path= "/Workspace/Repos/development/datavault-datamodelling/datascripts/tables/metadataregistry"

In [0]:
from dataclasses import dataclass

@dataclass
class RunNotebook:
    notebook_path: str
    parameters: Dict[str, Any]
    timeout_seconds: int = 3600
    result: Optional[Any] = None
    error: Optional[Any] = None
    status: Optional[Any] = None

    def run(self):
        try:
            self.result = dbutils.notebook.run(
                self.notebook_path,
                self.timeout_seconds,
                self.parameters
            )
            self.status = "success"
        except Exception as e:
            self.error = e
            self.status = "error"
        return self

@dataclass
class ExecuteSQLfiles:
    sql_files_path: str
    env: str = 'dev'
    catalog_env: str = 'dev'
    schema_name: str = 'metadataregistry'

    def execute_sql_files(self):
        sql_files = os.listdir(self.sql_files_path)
        for sql_file in sql_files:
            sql_file_path = os.path.join(self.sql_files_path, sql_file)
            with open(sql_file_path, 'r') as f:
                sql = f.read()
            sql = sql.replace('${catalog_name}', self.catalog_env)
            sql = sql.replace('${env}', self.env)
            sql = sql.replace('${schema_name}', self.schema_name)
            spark.sql(sql)
        return self

In [0]:
if __name__ == "__main__":
    # Run the notebook to create sql statements
    env = 'dev'
    catalog_env = 'dev'
    schema_name = 'metadataregistry'
    parameters = {
        "env": env,
        "catalog_env": catalog_env,
        "schema_name": schema_name
    }
    run_notebook = RunNotebook(
        notebook_path=notebook_path,
        parameters=parameters,
        timeout_seconds=1200  # (number of sql files * 20 seconds each)
    ).run()
    if run_notebook.status == "error":
        logger.error(f"Error running notebook: {run_notebook.error}")
        raise Exception(f"Error running notebook: {run_notebook.error}")

    # Execute the sql files
    try:
        execute_sqlfiles = ExecuteSQLfiles(
            sql_files_path=sql_files_path,
            env=env,
            catalog_env=catalog_env,
            schema_name=schema_name
        ).execute_sql_files()
    except Exception as e:
        logger.error(f"Error executing sql files: {e}")
        raise Exception(f"Error executing sql files: {e}")